<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_libero_transcoder_colab_jayden.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pi0.5 LIBERO Transcoder Simulation Notebook

Run Pi0.5 LIBERO rollouts in one of three modes: original model, probe with trained transcoders while preserving original actions, or replacement where transcoders substitute the action-expert MLPs.


## What This Runs

- `original`: standard Pi0.5 LIBERO rollout.
- `probe`: standard Pi0.5 actions, with transcoder latent activations saved for the action-expert MLPs.
- `replace`: action-expert MLP outputs are replaced by trained transcoder outputs, with latent activations saved.

Pi0.5 weights and LIBERO assets are still loaded from Hugging Face/cache. The transcoder checkpoint is a separate file loaded from Drive, Hugging Face, a URL, or a local notebook path.


In [ ]:
# @title Controls

# Drive folder shared only with approved collaborators.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
DRIVE_FOLDER_ID = "13z_Qh91Ww0jdXoju2kQsG12AaEPKZsnH"  # @param {type:"string"}
AUTO_CREATE_DRIVE_SHORTCUT = True  # @param {type:"boolean"}
APPROVED_ACCOUNTS = "programmer908@gmail.com"  # @param {type:"string"}
SHARED_HF_TOKEN_FILE = "secrets/HF_TOKEN.txt"  # @param {type:"string"}

# Repository.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

# Runtime preference. Pi0.5 policy loading is too memory-heavy for Colab T4.
REQUIRED_GPU = "L4"  # @param ["L4", "A100", "Any"]
MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}
MIN_SYSTEM_RAM_GB = 24  # @param {type:"integer"}

# Simulation controls.
RUN_MODE = "replace"  # @param ["original", "probe", "replace"]
RUN_LABEL = "replace-sanity"  # @param {type:"string"}
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10", "libero_spatial,libero_object,libero_goal,libero_10"]
TASK_IDS = "[0,1,2,3,4,5,6,7,8,9]"  # @param {type:"string"}
ANALYSIS_TASK_ID = 0  # @param {type:"integer"}
EPISODES = 3  # @param {type:"integer"}
EVAL_PROGRESS_SECONDS = 30  # @param {type:"integer"}

# Transcoder checkpoint. Required for probe/replace, ignored for original.
TRANSCODER_CHECKPOINT_SOURCE = "drive"  # @param ["drive", "hf", "url", "local"]
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}
TRANSCODER_LOCAL_PATH = ""  # @param {type:"string"}
TRANSCODER_HF_REPO = ""  # @param {type:"string"}
TRANSCODER_HF_FILE = ""  # @param {type:"string"}
TRANSCODER_URL = ""  # @param {type:"string"}
TRANSCODER_DTYPE = "auto"  # @param ["auto", "bfloat16", "float16", "float32"]

# Langfuse tracing controls. Store keys as private Colab Secrets, not in this notebook.
ENABLE_LANGFUSE_TRACE = True  # @param {type:"boolean"}
LANGFUSE_BASE_URL = "https://us.cloud.langfuse.com"  # @param {type:"string"}
LANGFUSE_ATTACH_IMAGES = True  # @param {type:"boolean"}
LANGFUSE_MAX_IMAGES = 2  # @param {type:"integer"}
LANGFUSE_MAX_MEDIA_BYTES = 2000000  # @param {type:"integer"}
LANGFUSE_CALLER = "marquise-colab"  # @param {type:"string"}
LANGFUSE_TENANT_ID = "libero-local"  # @param {type:"string"}
LANGFUSE_ENVIRONMENT = "development"  # @param {type:"string"}
LANGFUSE_TAGS = "transcoder-sanity,replace,libero"  # @param {type:"string"}

# Transcoder latent capture controls.
CAPTURE_TRANSCODER_LATENTS = True  # @param {type:"boolean"}
TRANSCODER_TOP_K = 64  # @param {type:"integer"}
TRANSCODER_MAX_CHUNKS = 1000  # @param {type:"integer"}
SAVE_FULL_LATENTS = False  # @param {type:"boolean"}
CAPTURE_TRANSCODER_DIFFUSION = True  # @param {type:"boolean"}

# Original activation capture controls.
CAPTURE_ACTIVATIONS = False  # @param {type:"boolean"}
CAPTURE_PARAM_STATS = False  # @param {type:"boolean"}
CAPTURE_MAX_CHUNKS = 1000  # @param {type:"integer"}
CAPTURE_LAYER_STRIDE = 1  # @param {type:"integer"}
CAPTURE_MAX_BINS = 64  # @param {type:"integer"}

# Colab report controls.
REPORT_MAX_ROWS = 80  # @param {type:"integer"}
GENERATE_DIAGNOSTIC_VIDEO = False  # @param {type:"boolean"}
DISPLAY_INDIVIDUAL_LAYER_GRAPHS = False  # @param {type:"boolean"}
LAYER_GRAPH_LIMIT = 6  # @param {type:"integer"}

# Cache behavior. Archive mode avoids slow Google Drive small-file copies.
CACHE_TRANSFER_MODE = "archive"  # @param ["archive", "folders"]
ALLOW_AUTH_REFRESH = True  # @param {type:"boolean"}
FORCE_AUTH_REFRESH = False  # @param {type:"boolean"}
HF_OFFLINE = True  # @param {type:"boolean"}

# Experimental prompt probe controls.
PROBE_POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}
PROBE_SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
PROBE_TASK_ID = 0  # @param {type:"integer"}
PROBE_LANGUAGE = "pick up the black bowl between the plate and the ramekin and place it on the plate"  # @param {type:"string"}
PROBE_SEED = 1000  # @param {type:"integer"}

# Aggregate transcoder feature-flow controls. This cell runs when executed; there is no run toggle.
TRANSCODER_PROBE_NAME = "transcoder-probe"  # @param {type:"string"}
RESET_TRANSCODER_PROBE_OUTPUT = True  # @param {type:"boolean"}
FEATURE_PROBE_EPISODES = "0,1,2,3,4"  # @param {type:"string"}
FEATURE_PROBE_BATCH_SIZE = 4  # @param {type:"integer"}
FEATURE_PROBE_NUM_WORKERS = 0  # @param {type:"integer"}
FEATURE_PROBE_COLLECTION_MODE = "inference"  # @param ["inference", "random-timestep", "training-forward"]
FEATURE_PROBE_NUM_INFERENCE_STEPS = 10  # @param {type:"integer"}
FEATURE_PROBE_NOISE_SAMPLES = 1  # @param {type:"integer"}
FEATURE_PROBE_TOP_K = 20  # @param {type:"integer"}
FEATURE_PROBE_TOP_M_ACTIVE = 100  # @param {type:"integer"}
FEATURE_PROBE_MAX_BATCHES = ""  # @param {type:"string"}
FEATURE_REPORT_MAX_FEATURES = 200  # @param {type:"integer"}
FEATURE_REPORT_TOP_EXAMPLES = 20  # @param {type:"integer"}
FEATURE_REPORT_SAVE_THUMBNAILS = True  # @param {type:"boolean"}
FEATURE_FLOW_TOP_FEATURES_PER_LAYER = 6  # @param {type:"integer"}

# Replacement equivalence controls. This estimates action error on identical observations.
ACTION_EQUIV_EPISODES = "0,1,2,3,4,5,6,7,8,9"  # @param {type:"string"}
ACTION_EQUIV_BATCH_SIZE = 2  # @param {type:"integer"}
ACTION_EQUIV_MAX_BATCHES = 50  # @param {type:"integer"}
ACTION_EQUIV_NUM_INFERENCE_STEPS = 10  # @param {type:"integer"}
# Counterfactual object-perturbation probe. Leave TARGET empty to only list the
# scene's objects, then fill it in and rerun.
COUNTERFACTUAL_TARGET = ""  # @param {type:"string"}
COUNTERFACTUAL_PLACEBO_TARGET = ""  # @param {type:"string"}
COUNTERFACTUAL_PERTURBATION = "blend"  # @param ["blend", "set", "hue"]
COUNTERFACTUAL_COLOR = "1.0,0.2,0.1"  # @param {type:"string"}
COUNTERFACTUAL_DOSE = "0.5,1.0"  # @param {type:"string"}
COUNTERFACTUAL_STATES = 2  # @param {type:"integer"}
COUNTERFACTUAL_STATE_STRIDE = 5  # @param {type:"integer"}
ACTION_EQUIV_CONTROL_BATCHES = 1  # @param {type:"integer"}
SANITY_COMPARE_MAKE_VIDEO = True  # @param {type:"boolean"}

print("Configured approved accounts:", APPROVED_ACCOUNTS)
print("Run mode:", RUN_MODE)
print("Langfuse trace:", ENABLE_LANGFUSE_TRACE)
print("Transcoder probe:", TRANSCODER_PROBE_NAME)


In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess
import time

from google.colab import drive


def parse_gib_from_meminfo() -> int:
    with open("/proc/meminfo", "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                return (kb + 1024 * 1024 - 1) // (1024 * 1024)
    return 0


def validate_colab_runtime():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        text=True,
        capture_output=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError("No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> GPU.")

    gpu_line = result.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",")]
    gpu_name = parts[0]
    gpu_mem_gb = int((int(parts[2]) + 1023) // 1024) if len(parts) >= 3 and parts[2].isdigit() else 0
    system_ram_gb = parse_gib_from_meminfo()
    print(f"Runtime memory: GPU={gpu_name} ~{gpu_mem_gb} GiB VRAM | system RAM ~{system_ram_gb} GiB")

    if REQUIRED_GPU != "Any" and REQUIRED_GPU.lower() not in gpu_name.lower():
        raise RuntimeError(
            f"Requested {REQUIRED_GPU}, but Colab allocated {gpu_name}. "
            "Use Runtime -> Change runtime type -> GPU -> L4/A100, then rerun from this cell."
        )
    if int(MIN_GPU_MEMORY_GB) > 0 and gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
        raise RuntimeError(
            f"GPU VRAM is too small for Pi0.5 LIBERO: {gpu_name} has ~{gpu_mem_gb} GiB, "
            f"need >= {MIN_GPU_MEMORY_GB} GiB. T4 usually exits 137 while loading the policy. "
            "Switch to L4 or A100 before running eval."
        )
    if int(MIN_SYSTEM_RAM_GB) > 0 and system_ram_gb < int(MIN_SYSTEM_RAM_GB):
        raise RuntimeError(
            f"System RAM is too small for Pi0.5 LIBERO: runtime has ~{system_ram_gb} GiB, "
            f"need >= {MIN_SYSTEM_RAM_GB} GiB. In Colab, choose a high-RAM L4/A100 runtime."
        )


validate_colab_runtime()


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient.

    Colab raises a bare ``ValueError: mount failed`` that says nothing about the
    cause. By far the most common one here is other live Colab sessions each
    holding their own Drive mount -- easy to hit when the original/probe and
    replace notebooks are open at the same time.
    """
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return

    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            # force_remount clears a half-established mount left by a failed try.
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:  # Colab raises a bare ValueError here.
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)

    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "This notebook needs Drive: the transcoder checkpoint, HF cache and outputs all live there.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions are holding Drive mounts. Runtime > Manage sessions, "
        "terminate the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked or dismissed. Allow popups and third-party "
        "cookies for colab.research.google.com, then rerun.\n"
        "  3. A transient Drive outage. Restart the runtime and try again in a few minutes."
    ) from last_error


mount_drive_with_retry()


def ensure_drive_root_visible(root_value: str) -> Path:
    root = Path(root_value)
    if root.exists():
        return root

    folder_id = str(globals().get("DRIVE_FOLDER_ID", "")).strip()
    auto_shortcut = bool(globals().get("AUTO_CREATE_DRIVE_SHORTCUT", True))
    mydrive_prefixes = ("/content/drive/MyDrive/", "/content/drive/My Drive/")
    if folder_id and auto_shortcut and str(root).startswith(mydrive_prefixes):
        print("Drive root is not visible yet; creating a My Drive shortcut to the shared folder.", flush=True)
        try:
            from google.colab import auth
            auth.authenticate_user()
            try:
                from googleapiclient.discovery import build
            except Exception:
                subprocess.run(["python3", "-m", "pip", "install", "-q", "-U", "google-api-python-client"], check=True)
                from googleapiclient.discovery import build
            service = build("drive", "v3")
            metadata = {
                "name": root.name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": folder_id},
                "parents": ["root"],
            }
            created = service.files().create(body=metadata, fields="id,name").execute()
            print(f"Created Drive shortcut: {created.get('name')} ({created.get('id')})", flush=True)
            for _ in range(12):
                if root.exists():
                    return root
                time.sleep(5)
        except Exception as exc:
            raise RuntimeError(
                "Could not auto-create the Drive shortcut. Confirm the folder was shared with this Google account "
                "and DRIVE_FOLDER_ID is the shared folder ID."
            ) from exc
        if root.exists():
            return root
        raise RuntimeError(
            f"Drive shortcut was created, but {root} is not visible in the mounted filesystem yet. "
            "Rerun this cell, or use a Google Workspace Shared Drive path."
        )

    if folder_id and str(root).startswith("/content/drive/Shareddrives/"):
        raise RuntimeError(
            f"Shared Drive path is not visible: {root}. Confirm the teammate has access to the Shared Drive."
        )

    if not folder_id:
        print(
            "DRIVE_FOLDER_ID is blank and DRIVE_ROOT does not exist. Creating a new folder at DRIVE_ROOT; "
            "this is intended only for initial owner setup. For zero teammate setup, set DRIVE_FOLDER_ID in the notebook.",
            flush=True,
        )
    return root


DRIVE_ROOT = ensure_drive_root_visible(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "hf_home"
DRIVE_LIBERO_CACHE = DRIVE_ROOT / "libero_cache"
DRIVE_LIBERO_DATASETS = DRIVE_ROOT / "libero_datasets"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
DRIVE_SECRETS = DRIVE_ROOT / "secrets"
DRIVE_HF_TOKEN_FILE = DRIVE_ROOT / SHARED_HF_TOKEN_FILE
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_LIBERO_CACHE, DRIVE_LIBERO_DATASETS, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META, DRIVE_SECRETS]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "hf_home.tar"
LIBERO_CACHE_ARCHIVE = DRIVE_ARCHIVES / "libero_cache.tar"
LIBERO_DATASETS_ARCHIVE = DRIVE_ARCHIVES / "libero_datasets.tar"

print("Drive root:", DRIVE_ROOT)
print("Archive dir:", DRIVE_ARCHIVES)
print("Expected sharing: restricted to", APPROVED_ACCOUNTS)
print("Optional shared HF token file:", DRIVE_HF_TOKEN_FILE)


In [ ]:
# @title Install Native Runtime Equivalent To cloud/libero/Dockerfile

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    try:
        return subprocess.run(cmd, env=env, check=True)
    except subprocess.CalledProcessError as exc:
        print(f"Command failed with exit code {exc.returncode}: {' '.join(cmd)}", flush=True)
        raise


def install_uv() -> str:
    # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
    # Use a notebook-owned binary so every clean runtime follows the same path.
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    installer = Path("/tmp/install-uv.sh")
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", installer], env=env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
    run([UV, "--version"])
    return UV


apt_packages = [
    "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
    "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
    "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
]
apt_env = os.environ.copy()
apt_env["DEBIAN_FRONTEND"] = "noninteractive"
run(["apt-get", "update", "-qq"], env=apt_env)
run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
UV = install_uv()

# Managed Python avoids system-package/venv quirks in Colab images.
run([UV, "python", "install", "3.12"])
run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])

run([
    UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
    "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy", "langfuse",
])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
print("Python:", PYTHON)
run([str(PYTHON), "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"])


In [ ]:
 # @title Clone Or Update Repo

from pathlib import Path
import subprocess

LOCAL_REPO = Path("/content/groot-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)

print("Repo:", LOCAL_REPO)
subprocess.run(["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import os
import shutil
import shlex
import subprocess
import time

LOCAL_HF_HOME = Path("/content/hf_home")
LOCAL_LIBERO_CACHE = Path.home() / ".cache/libero"
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/pi05_libero" / RUN_LABEL
LOCAL_DATA_ROOT = LOCAL_REPO / "data/libero/datasets"
LOCAL_LIBERO_CONFIG = LOCAL_REPO / ".libero"
for path in [LOCAL_OUTPUT_ROOT, LOCAL_LIBERO_CONFIG]:
    path.mkdir(parents=True, exist_ok=True)

HF_CACHE_REFRESHED = False

repos = ["lerobot/pi05_libero_finetuned", "google/paligemma-3b-pt-224", "lerobot/libero-assets"]


def hf_repo_type(repo_id: str) -> str:
    return "dataset" if repo_id == "lerobot/libero-assets" else "model"


def now_text() -> str:
    return time.strftime("%H:%M:%S")


def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def timed(label, fn):
    print(f"\n== {label} | start {now_text()} ==", flush=True)
    start = time.time()
    result = fn()
    elapsed = time.time() - start
    print(f"== done: {label} | elapsed {format_duration(elapsed)} | finish {now_text()} ==", flush=True)
    return result


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    start = time.time()
    result = subprocess.run([str(x) for x in cmd], check=True, **kwargs)
    print(f"+ done in {format_duration(time.time() - start)}", flush=True)
    return result


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def dir_has_anything(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def hf_cache_has(repo_id: str, root: Path) -> bool:
    namespace, name = repo_id.split("/", 1)
    prefix = "datasets" if hf_repo_type(repo_id) == "dataset" else "models"
    return (root / "hub" / f"{prefix}--{namespace}--{name}").exists()


def get_hf_token(required: bool = False) -> str:
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
        if token:
            print("Using private Colab Secret HF_TOKEN.", flush=True)
            return token
    except Exception:
        pass
    token_file = globals().get("DRIVE_HF_TOKEN_FILE")
    if token_file and Path(token_file).exists():
        token = Path(token_file).read_text().strip()
        if token:
            print(f"Using shared Drive HF token file: {token_file}", flush=True)
            return token
    if required:
        raise RuntimeError(
            "ALLOW_AUTH_REFRESH=True requires either private Colab Secret HF_TOKEN "
            f"or shared Drive token file at {globals().get('DRIVE_HF_TOKEN_FILE', 'DRIVE_ROOT/secrets/HF_TOKEN.txt')}."
        )
    return ""



def path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def du(path: Path):
    if path.exists():
        run(["du", "-sh", path])


def extract_tar(archive: Path, dst: Path):
    reset_dir(dst)
    size = archive.stat().st_size
    print(f"Archive: {archive} ({format_bytes(size)}) -> {dst}", flush=True)
    cmd = f"set -euo pipefail; pv -ptebarf -s {size} {shlex.quote(str(archive))} | tar -C {shlex.quote(str(dst))} -xf -"
    run(["bash", "-lc", cmd])
    du(dst)


def create_tar(src: Path, archive: Path):
    if not dir_has_anything(src):
        print(f"Skipping archive for empty directory: {src}", flush=True)
        return
    archive.parent.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(archive.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    size_bytes = path_size_bytes(src)
    print(f"Archiving: {src} ({format_bytes(size_bytes)}) -> {archive}", flush=True)
    cmd = f"set -euo pipefail; tar -C {shlex.quote(str(src))} -cf - . | pv -ptebarf -s {size_bytes} > {shlex.quote(str(tmp))}"
    run(["bash", "-lc", cmd])
    tmp.replace(archive)
    run(["ls", "-lh", archive])


def rsync_tree(src: Path, dst: Path, reset: bool = True):
    src.mkdir(parents=True, exist_ok=True)
    if reset:
        reset_dir(dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
    print(f"Rsync: {src} ({format_bytes(path_size_bytes(src))}) -> {dst}", flush=True)
    run(["rsync", "-a", "--human-readable", "--info=progress2", "--stats", f"{src}/", f"{dst}/"])
    du(dst)


if CACHE_TRANSFER_MODE == "archive" and HF_HOME_ARCHIVE.exists() and not FORCE_AUTH_REFRESH:
    timed("extract HF cache archive from Drive to /content", lambda: extract_tar(HF_HOME_ARCHIVE, LOCAL_HF_HOME))
elif ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=True)
    reset_dir(LOCAL_HF_HOME)
    run([str(PYTHON), "-c", "import huggingface_hub; print('huggingface_hub OK')"])
    refresh_code = """
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import enable_progress_bars
import os
import subprocess
import threading
import time

repos = os.environ['REFRESH_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'
heartbeat_seconds = int(os.environ.get('HF_PROGRESS_HEARTBEAT_SECONDS', '15'))


def now_text():
    return time.strftime('%H:%M:%S')


def format_duration(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return 'unknown'
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f'{hours}h{minutes:02d}m{secs:02d}s'
    if minutes:
        return f'{minutes}m{secs:02d}s'
    return f'{secs}s'


def format_bytes(value):
    if value is None:
        return 'unknown'
    value = float(value)
    units = ['B', 'KiB', 'MiB', 'GiB', 'TiB']
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f'{value:.1f}{unit}' if unit != 'B' else f'{value:.0f}{unit}'
        value /= 1024
    return f'{value:.1f}PiB'


def path_size_bytes(path):
    if not os.path.exists(path):
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not os.path.islink(os.path.join(root, name))]
        for name in files:
            file_path = os.path.join(root, name)
            try:
                total += os.lstat(file_path).st_size
            except OSError:
                pass
    return total


def estimate_repo_size(api, repo):
    try:
        if repo_type(repo) == 'dataset':
            info = api.dataset_info(repo_id=repo, token=token, files_metadata=True)
        else:
            info = api.model_info(repo_id=repo, token=token, files_metadata=True)
        sizes = [getattr(sibling, 'size', None) for sibling in getattr(info, 'siblings', [])]
        total = sum(size for size in sizes if isinstance(size, int))
        count = len(sizes)
        return count, total or None
    except Exception as exc:
        print(f'[{repo}] could not estimate size before download: {type(exc).__name__}: {exc}', flush=True)
        return None, None


def heartbeat(repo, root, start_bytes, estimated_total, stop_event):
    last_time = time.time()
    last_bytes = start_bytes
    start_time = last_time
    while not stop_event.wait(heartbeat_seconds):
        now = time.time()
        current_bytes = path_size_bytes(root)
        delta = max(0, current_bytes - start_bytes)
        recent_rate = max(0, current_bytes - last_bytes) / max(0.001, now - last_time)
        avg_rate = delta / max(0.001, now - start_time)
        if estimated_total and avg_rate > 0:
            remaining = max(0, estimated_total - delta)
            eta = remaining / avg_rate
            pct = min(100.0, (delta / estimated_total) * 100.0)
            estimate_text = f'{pct:5.1f}% of est. {format_bytes(estimated_total)} | ETA {format_duration(eta)}'
        else:
            estimate_text = 'ETA unknown'
        print(
            f'[{repo}] progress {format_bytes(delta)} | recent {format_bytes(recent_rate)}/s | '
            f'avg {format_bytes(avg_rate)}/s | elapsed {format_duration(now - start_time)} | {estimate_text}',
            flush=True,
        )
        last_time = now
        last_bytes = current_bytes


os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['TQDM_DISABLE'] = '0'
os.environ['TQDM_MININTERVAL'] = '1'
try:
    enable_progress_bars()
except Exception as exc:
    print(f'Could not force-enable Hugging Face progress bars: {type(exc).__name__}: {exc}', flush=True)

api = HfApi()
for index, repo in enumerate(repos, 1):
    print(f'\\n[{index}/{len(repos)}] preparing {repo} at {now_text()}', flush=True)
    file_count, estimated_total = estimate_repo_size(api, repo)
    if file_count is not None:
        print(f'[{repo}] estimated remote files={file_count} size={format_bytes(estimated_total)}', flush=True)
    before = path_size_bytes(cache_dir)
    stop_event = threading.Event()
    monitor = threading.Thread(target=heartbeat, args=(repo, cache_dir, before, estimated_total, stop_event), daemon=True)
    start = time.time()
    monitor.start()
    try:
        path = snapshot_download(
            repo_id=repo,
            repo_type=repo_type(repo),
            cache_dir=cache_dir,
            token=token,
            max_workers=8,
        )
    finally:
        stop_event.set()
        monitor.join(timeout=2)
    elapsed = time.time() - start
    after = path_size_bytes(cache_dir)
    delta = max(0, after - before)
    avg_rate = delta / max(0.001, elapsed)
    print(f'[{repo}] -> {path}', flush=True)
    print(
        f'[{repo}] done at {now_text()} | elapsed {format_duration(elapsed)} | '
        f'cache delta {format_bytes(delta)} | avg {format_bytes(avg_rate)}/s | total cache {format_bytes(after)}',
        flush=True,
    )
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token,
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "REFRESH_REPOS": ",".join(repos),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
        "HF_PROGRESS_HEARTBEAT_SECONDS": "15",
    })
    timed("download gated HF models to fast local disk", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
elif CACHE_TRANSFER_MODE == "folders" or dir_has_anything(DRIVE_HF_HOME):
    print("Archive missing; falling back to Drive folder rsync. This can be very slow for HF caches.", flush=True)
    timed("copy HF cache folder from Drive to /content", lambda: rsync_tree(DRIVE_HF_HOME, LOCAL_HF_HOME))
else:
    raise RuntimeError(
        "No HF cache archive or folder found in Drive. Set ALLOW_AUTH_REFRESH=True once with private Colab Secret HF_TOKEN or DRIVE_ROOT/secrets/HF_TOKEN.txt."
    )

missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]
if missing_local and ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=False)
    refresh_missing_code = """
from huggingface_hub import snapshot_download
import os

repos = os.environ['MISSING_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in repos:
    print('refreshing missing cache entry', repo, 'repo_type', repo_type(repo), flush=True)
    path = snapshot_download(
        repo_id=repo,
        repo_type=repo_type(repo),
        cache_dir=cache_dir,
        token=token,
        max_workers=8,
    )
    print(repo, '->', path, flush=True)
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token or "",
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "MISSING_REPOS": ",".join(missing_local),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
    })
    timed("refresh missing HF cache entries", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_missing_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
    missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]

if missing_local:
    raise RuntimeError(
        "Local HF cache is missing: " + ", ".join(missing_local) + "\n"
        "Run once with ALLOW_AUTH_REFRESH=True plus private Colab Secret HF_TOKEN or DRIVE_ROOT/secrets/HF_TOKEN.txt, "
        "or populate DRIVE_ROOT/archives/hf_home.tar."
    )

if CACHE_TRANSFER_MODE == "archive" and LIBERO_CACHE_ARCHIVE.exists():
    timed("extract LIBERO cache archive from Drive", lambda: extract_tar(LIBERO_CACHE_ARCHIVE, LOCAL_LIBERO_CACHE))
else:
    timed("copy LIBERO cache folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_CACHE, LOCAL_LIBERO_CACHE))

if CACHE_TRANSFER_MODE == "archive" and LIBERO_DATASETS_ARCHIVE.exists():
    timed("extract LIBERO datasets archive from Drive", lambda: extract_tar(LIBERO_DATASETS_ARCHIVE, LOCAL_DATA_ROOT))
else:
    timed("copy LIBERO datasets folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_DATASETS, LOCAL_DATA_ROOT))

os.environ.update({
    "PATH": f"{VENV / 'bin'}:" + os.environ["PATH"],
    "PYTHON": str(PYTHON),
    "HF_HOME": str(LOCAL_HF_HOME),
    "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HF_XET_HIGH_PERFORMANCE": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "MPLBACKEND": "Agg",
    "LIBERO_CONFIG_PATH": str(LOCAL_LIBERO_CONFIG),
    "LIBERO_DATASET_DIR": str(LOCAL_DATA_ROOT),
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
})
if HF_OFFLINE:
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

validate_code = """
from huggingface_hub import snapshot_download
import os


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in os.environ['VALIDATE_REPOS'].split(','):
    path = snapshot_download(repo_id=repo, repo_type=repo_type(repo), cache_dir=os.environ['HF_HUB_CACHE'], local_files_only=True)
    print(repo, '->', path, flush=True)
"""
env = os.environ.copy()
env["VALIDATE_REPOS"] = ",".join(repos)
timed("validate local HF cache with local_files_only=True", lambda: subprocess.run([str(PYTHON), "-u", "-c", validate_code], env=env, check=True))

install_assets_code = """
from pathlib import Path
import os
import shutil
import site

from huggingface_hub import snapshot_download

roots = [Path(path) for path in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
for root in roots:
    candidate = root / "libero" / "libero"
    if candidate.exists():
        libero_root = candidate
        break
else:
    raise SystemExit("Could not find installed libero package path")

assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
snapshot = Path(snapshot_download(
    repo_id="lerobot/libero-assets",
    repo_type="dataset",
    cache_dir=os.environ["HF_HUB_CACHE"],
    local_files_only=True,
))
if not required.exists():
    print(f"Installing LIBERO assets: {snapshot} -> {assets_dir}", flush=True)
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snapshot.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
if not required.exists():
    raise SystemExit(f"LIBERO asset install failed; missing {required}")
print("libero_assets OK", required, flush=True)
"""
timed("install LIBERO assets into package", lambda: subprocess.run([str(PYTHON), "-u", "-c", install_assets_code], env=os.environ.copy(), check=True))

print("\nLocal cache summary:")
du(LOCAL_HF_HOME)
du(LOCAL_LIBERO_CACHE)
du(LOCAL_DATA_ROOT)


In [ ]:
# @title Load Langfuse Credentials From Colab Secrets

import os

# Do not paste API keys into the notebook. Store these as private Colab Secrets:
# LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, and optionally LANGFUSE_BASE_URL.
def load_langfuse_env():
    if not ENABLE_LANGFUSE_TRACE:
        os.environ["PI05_LANGFUSE_TRACE"] = "0"
        print("Langfuse tracing disabled by ENABLE_LANGFUSE_TRACE=False")
        return

    os.environ["PI05_LANGFUSE_TRACE"] = "1"
    os.environ["LANGFUSE_BASE_URL"] = (LANGFUSE_BASE_URL or "https://us.cloud.langfuse.com").strip()
    os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]
    os.environ["PI05_LANGFUSE_MEDIA"] = "1" if LANGFUSE_ATTACH_IMAGES else "0"
    os.environ["PI05_LANGFUSE_MAX_IMAGES"] = str(LANGFUSE_MAX_IMAGES)
    os.environ["PI05_LANGFUSE_MAX_MEDIA_BYTES"] = str(LANGFUSE_MAX_MEDIA_BYTES)
    os.environ["PI05_LANGFUSE_CALLER"] = LANGFUSE_CALLER.strip() or "colab"
    os.environ["PI05_LANGFUSE_TENANT_ID"] = LANGFUSE_TENANT_ID.strip() or "libero-local"
    os.environ["PI05_LANGFUSE_ENVIRONMENT"] = LANGFUSE_ENVIRONMENT.strip() or "development"
    os.environ["PI05_LANGFUSE_TAGS"] = LANGFUSE_TAGS.strip()

    missing = []
    try:
        from google.colab import userdata
    except Exception:
        userdata = None

    for name in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]:
        if os.environ.get(name):
            continue
        value = ""
        if userdata is not None:
            try:
                value = userdata.get(name) or ""
            except Exception:
                value = ""
        if value:
            os.environ[name] = value
        else:
            missing.append(name)

    if missing:
        print("Langfuse tracing requested, but missing Colab Secrets:", ", ".join(missing))
        print("The run will still write local artifacts; Langfuse sends will be disabled by the helper until keys exist.")
    else:
        print("Langfuse tracing configured for:", os.environ["LANGFUSE_BASE_URL"])


load_langfuse_env()


In [ ]:
# @title Resolve Transcoder Checkpoint

from pathlib import Path
import os
import subprocess

TRANSCODER_CHECKPOINT = ""

if RUN_MODE == "original":
    print("RUN_MODE=original: no transcoder checkpoint is needed.")
else:
    source = TRANSCODER_CHECKPOINT_SOURCE.strip().lower()
    checkpoint_dir = LOCAL_REPO / "checkpoints" / "transcoders"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    if source == "drive":
        candidate = Path(TRANSCODER_DRIVE_PATH)
        if not candidate.is_absolute():
            candidate = DRIVE_ROOT / candidate
        TRANSCODER_CHECKPOINT = str(candidate)
    elif source == "local":
        TRANSCODER_CHECKPOINT = TRANSCODER_LOCAL_PATH.strip()
    elif source == "hf":
        if not TRANSCODER_HF_REPO.strip() or not TRANSCODER_HF_FILE.strip():
            raise ValueError("Set TRANSCODER_HF_REPO and TRANSCODER_HF_FILE for source='hf'.")
        from huggingface_hub import hf_hub_download
        token = get_hf_token(required=not HF_OFFLINE)
        TRANSCODER_CHECKPOINT = hf_hub_download(
            repo_id=TRANSCODER_HF_REPO.strip(),
            filename=TRANSCODER_HF_FILE.strip(),
            cache_dir=os.environ["HF_HUB_CACHE"],
            local_files_only=bool(HF_OFFLINE),
            token=token or None,
        )
    elif source == "url":
        if not TRANSCODER_URL.strip():
            raise ValueError("Set TRANSCODER_URL for source='url'.")
        out = checkpoint_dir / Path(TRANSCODER_URL.split("?", 1)[0]).name
        if not out.exists():
            subprocess.run(["curl", "-L", TRANSCODER_URL, "-o", str(out)], check=True)
        TRANSCODER_CHECKPOINT = str(out)
    else:
        raise ValueError(f"Unknown TRANSCODER_CHECKPOINT_SOURCE={TRANSCODER_CHECKPOINT_SOURCE!r}")

    checkpoint_path = Path(TRANSCODER_CHECKPOINT)
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Transcoder checkpoint not found: {checkpoint_path}. "
            "Use source='drive' with a valid shared Drive path, or source='hf'/'url' once the checkpoint is hosted."
        )
    print("Transcoder checkpoint:", checkpoint_path)
    print("Size:", f"{checkpoint_path.stat().st_size / (1024 ** 3):.2f} GiB")


In [ ]:
# @title Run LIBERO Eval With Progress

import ast
import json
import os
import re
import subprocess
import sys
import threading
import time
from pathlib import Path

run_env = os.environ.copy()
run_id = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
run_dir = LOCAL_OUTPUT_ROOT / run_id
suffix = 1
while run_dir.exists():
    run_id = f"{run_id}-{suffix}"
    run_dir = LOCAL_OUTPUT_ROOT / run_id
    suffix += 1
run_dir.mkdir(parents=True, exist_ok=True)

pythonpath_parts = [
    str(LOCAL_REPO / "cloud/libero/transcoder_runtime"),
    str(LOCAL_REPO / "cloud/libero/activation_capture"),
    str(LOCAL_REPO / "src"),
    str(LOCAL_REPO / "scripts"),
]
if run_env.get("PYTHONPATH"):
    pythonpath_parts.append(run_env["PYTHONPATH"])

run_env.update({
    "PYTHON": str(PYTHON),
    "DEVICE": "cuda",
    "DTYPE": "bfloat16",
    "MIN_GPU_MEM_GB": str(MIN_GPU_MEMORY_GB),
    "MIN_HOST_RAM_GB": str(MIN_SYSTEM_RAM_GB),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
    "TASK_IDS": TASK_IDS.strip(),
    "RUN_ID": run_id,
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
    "CAPTURE_ACTIVATIONS": "1" if CAPTURE_ACTIVATIONS else "0",
    "CAPTURE_PARAM_STATS": "1" if CAPTURE_PARAM_STATS else "0",
    "CAPTURE_MAX_CHUNKS": str(CAPTURE_MAX_CHUNKS),
    "CAPTURE_LAYER_STRIDE": str(CAPTURE_LAYER_STRIDE),
    "CAPTURE_MAX_BINS": str(CAPTURE_MAX_BINS),
    "PYTHONPATH": ":".join(pythonpath_parts),
    "PI05_TRANSCODER_MODE": RUN_MODE,
    "PI05_TRANSCODER_CHECKPOINT": TRANSCODER_CHECKPOINT,
    "PI05_TRANSCODER_CAPTURE_DIR": str(run_dir / "transcoder_capture"),
    "PI05_TRANSCODER_CAPTURE_LATENTS": "1" if CAPTURE_TRANSCODER_LATENTS else "0",
    "PI05_TRANSCODER_CAPTURE_DIFFUSION": "1" if CAPTURE_TRANSCODER_DIFFUSION else "0",
    "PI05_TRANSCODER_TOP_K": str(TRANSCODER_TOP_K),
    "PI05_TRANSCODER_MAX_CHUNKS": str(TRANSCODER_MAX_CHUNKS if (CAPTURE_TRANSCODER_LATENTS or CAPTURE_TRANSCODER_DIFFUSION) else 0),
    "PI05_TRANSCODER_SAVE_FULL_LATENTS": "1" if SAVE_FULL_LATENTS else "0",
    "PI05_TRANSCODER_DTYPE": TRANSCODER_DTYPE,
    "PI05_N_ACTION_STEPS": "10",
    "PI05_LANGFUSE_TRACE": "1" if ENABLE_LANGFUSE_TRACE else "0",
    "PI05_LANGFUSE_MEDIA": "1" if LANGFUSE_ATTACH_IMAGES else "0",
    "PI05_LANGFUSE_MAX_IMAGES": str(LANGFUSE_MAX_IMAGES),
    "PI05_LANGFUSE_MAX_MEDIA_BYTES": str(LANGFUSE_MAX_MEDIA_BYTES),
    "PI05_LANGFUSE_CALLER": LANGFUSE_CALLER.strip() or "colab",
    "PI05_LANGFUSE_TENANT_ID": LANGFUSE_TENANT_ID.strip() or "libero-local",
    "PI05_LANGFUSE_SESSION_ID": os.environ.get("PI05_LANGFUSE_SESSION_ID", run_id),
    "PI05_LANGFUSE_ENVIRONMENT": LANGFUSE_ENVIRONMENT.strip() or "development",
    "PI05_LANGFUSE_TAGS": LANGFUSE_TAGS.strip(),
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL", LANGFUSE_BASE_URL),
    "LANGFUSE_HOST": os.environ.get("LANGFUSE_HOST", os.environ.get("LANGFUSE_BASE_URL", LANGFUSE_BASE_URL)),
    "PYTHONUNBUFFERED": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MPLBACKEND": "Agg",
})

if RUN_MODE in {"probe", "replace"} and not TRANSCODER_CHECKPOINT:
    raise RuntimeError("Resolve the transcoder checkpoint before running probe/replace mode.")

cmd = ["bash", "cloud/libero/run_pi05_libero.sh", SUITE, str(EPISODES)]
launcher_log = run_dir / "colab_launcher.log"

ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def eval_format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def eval_format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def eval_path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def clean_log_text(text: str) -> str:
    return ANSI_RE.sub("", text).replace("\r", "\n")


def tail_text(path: Path, max_bytes: int = 240_000) -> str:
    if not path.exists():
        return ""
    with path.open("rb") as handle:
        try:
            handle.seek(-max_bytes, os.SEEK_END)
        except OSError:
            handle.seek(0)
        return handle.read().decode("utf-8", errors="replace")


def task_id_count(raw: str) -> int | None:
    raw = raw.strip()
    if not raw:
        return None
    try:
        value = ast.literal_eval(raw)
    except Exception:
        return None
    if isinstance(value, int):
        return 1
    if isinstance(value, (list, tuple, set)):
        return len(value)
    return None


def expected_eval_count() -> int:
    suites = [item.strip() for item in SUITE.split(",") if item.strip()]
    per_suite = task_id_count(TASK_IDS)
    if per_suite is None:
        per_suite = 10
    return max(1, len(suites) * max(1, per_suite) * max(1, int(EPISODES)))


def parse_progress_line(text: str) -> dict[str, str]:
    clean = clean_log_text(text)
    result: dict[str, str] = {}
    try:
        rollout_matches = list(re.finditer(
            r"Running rollout with at most\s+(\d+)\s+steps:\s+(\d+)%.*?\|\s*(\d+)/(?:\s*)?(\d+).*?running_success_rate=([0-9.]+)%",
            clean,
        ))
        if rollout_matches:
            match = rollout_matches[-1]
            _max_steps, pct, step, total, success = match.groups()
            result["rollout"] = f"step {step}/{total} ({pct}%), running success {success}%"
        batch_matches = list(re.finditer(
            r"Stepping through eval batches:\s+(\d+)%.*?\|\s*(\d+)/(?:\s*)?(\d+)",
            clean,
        ))
        if batch_matches:
            match = batch_matches[-1]
            pct, done, total = match.groups()
            result["batch"] = f"batch {done}/{total} ({pct}%)"
    except Exception as exc:
        result["parse_error"] = f"progress parser skipped: {type(exc).__name__}: {exc}"
    for line in reversed([line.strip() for line in clean.splitlines() if line.strip()]):
        if any(key in line for key in ["Running rollout", "Stepping through eval batches", "hf_access OK", "torch ", "cuda_available", "Overall Aggregated Metrics", "End of eval", "Saved results", "Traceback", "Error", "Exception"]):
            result["last"] = line[-260:]
            break
    return result


def activation_chunk_count() -> int:
    events = run_dir / "activation_capture" / "events.jsonl"
    if events.exists():
        try:
            data = tail_text(events, max_bytes=max(events.stat().st_size, 1))
            return data.count('"type":"chunk_start"')
        except OSError:
            pass
    image_dir = run_dir / "activation_capture" / "images"
    if image_dir.exists():
        chunks = set()
        for path in image_dir.glob("chunk_*_*"):
            parts = path.name.split("_")
            if len(parts) > 1:
                chunks.add(parts[1])
        return len(chunks)
    return 0


def print_eval_progress(start_time: float, final: bool = False, failed: bool = False):
    text = tail_text(launcher_log) + "\n" + tail_text(run_dir / "run.log")
    parsed = parse_progress_line(text)
    videos = list((run_dir / "videos").glob("**/*.mp4")) if (run_dir / "videos").exists() else []
    expected = expected_eval_count()
    video_pct = min(100.0, 100.0 * len(videos) / expected) if expected else 0.0
    chunks = activation_chunk_count() if CAPTURE_ACTIVATIONS else 0
    output_size = eval_format_bytes(eval_path_size_bytes(run_dir))
    parts = [
        f"elapsed {eval_format_duration(time.time() - start_time)}",
        f"videos {len(videos)}/{expected} ({video_pct:.1f}%)",
        f"output {output_size}",
    ]
    if CAPTURE_ACTIVATIONS:
        cap = int(CAPTURE_MAX_CHUNKS)
        parts.append(f"activation chunks {chunks}/{cap if cap > 0 else 'unlimited'}")
    if parsed.get("batch"):
        parts.append(parsed["batch"])
    if parsed.get("rollout"):
        parts.append(parsed["rollout"])
    prefix = "\n[eval stopped] " if failed else ("\n[eval complete] " if final else "\n[eval progress] ")
    print(prefix + " | ".join(parts), flush=True)
    if parsed.get("parse_error"):
        print("[eval parser] " + parsed["parse_error"], flush=True)
    if parsed.get("last"):
        print("[eval last] " + parsed["last"], flush=True)


def print_failure_diagnostics(returncode: int):
    print(f"\nEval failed with exit code {returncode}", flush=True)
    print(f"Run dir: {run_dir}", flush=True)
    print(f"Launcher log: {launcher_log}", flush=True)
    print(f"Eval log: {run_dir / 'run.log'}", flush=True)
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except Exception as exc:
        print(f"Could not run nvidia-smi: {exc}", flush=True)
    for label, path in [("launcher log", launcher_log), ("eval run.log", run_dir / "run.log")]:
        if not path.exists():
            print(f"\n--- no {label} yet ---", flush=True)
            continue
        lines = [line for line in clean_log_text(tail_text(path, 80_000)).splitlines() if line.strip()]
        print(f"\n--- last {min(80, len(lines))} lines from {label} ---", flush=True)
        for line in lines[-80:]:
            print(line[-500:], flush=True)


validate_colab_runtime()

print("Running:", " ".join(cmd), flush=True)
print("Run dir:", run_dir, flush=True)
eval_progress_seconds = max(5, int(globals().get("EVAL_PROGRESS_SECONDS", 30)))
print("Progress heartbeat seconds:", eval_progress_seconds, flush=True)
start = time.time()
with launcher_log.open("wb") as log_handle:
    process = subprocess.Popen(
        cmd,
        cwd=LOCAL_REPO,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    def write_stdout(chunk: bytes):
        buffer = getattr(sys.stdout, "buffer", None)
        if buffer is not None:
            buffer.write(chunk)
            buffer.flush()
        else:
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
            sys.stdout.flush()

    def stream_output():
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            write_stdout(chunk)
            log_handle.write(chunk)
            log_handle.flush()

    stream_thread = threading.Thread(target=stream_output, daemon=True)
    stream_thread.start()
    while process.poll() is None:
        time.sleep(eval_progress_seconds)
        if process.poll() is None:
            print_eval_progress(start)
    stream_thread.join(timeout=10)
    returncode = process.returncode

print_eval_progress(start, final=(returncode == 0), failed=(returncode != 0))
if returncode != 0:
    print_failure_diagnostics(returncode)
    raise subprocess.CalledProcessError(returncode, cmd)

LATEST_RUN = run_dir
print("Latest run:", LATEST_RUN)


In [ ]:
# @title Persist Outputs And Caches Back To Drive

# Outputs are small enough to sync as folders and should be visible directly in Drive.
rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)

# Persist caches as tar archives to avoid slow Drive small-file copies on the next Colab session.
if CACHE_TRANSFER_MODE == "archive":
    if FORCE_AUTH_REFRESH or globals().get("HF_CACHE_REFRESHED") or not HF_HOME_ARCHIVE.exists():
        timed("write HF cache archive to Drive", lambda: create_tar(LOCAL_HF_HOME, HF_HOME_ARCHIVE))
    timed("write LIBERO cache archive to Drive", lambda: create_tar(LOCAL_LIBERO_CACHE, LIBERO_CACHE_ARCHIVE))
    timed("write LIBERO datasets archive to Drive", lambda: create_tar(LOCAL_DATA_ROOT, LIBERO_DATASETS_ARCHIVE))
else:
    timed("sync unpacked HF cache to Drive", lambda: rsync_tree(LOCAL_HF_HOME, DRIVE_HF_HOME))
    timed("sync unpacked LIBERO cache to Drive", lambda: rsync_tree(LOCAL_LIBERO_CACHE, DRIVE_LIBERO_CACHE))
    timed("sync unpacked LIBERO datasets to Drive", lambda: rsync_tree(LOCAL_DATA_ROOT, DRIVE_LIBERO_DATASETS))

print("Persisted outputs to:", DRIVE_OUTPUTS)
print("Cache archive dir:", DRIVE_ARCHIVES)

In [ ]:
# @title Display Summary, Videos, And Activation Report Inline

import json
import subprocess
from pathlib import Path
from IPython.display import display, Video, Markdown, Image, HTML

run_dir = LATEST_RUN
info_path = run_dir / "eval_info.json"
if info_path.exists():
    info = json.loads(info_path.read_text())
    overall = info.get("overall", {})
else:
    info = {}
    overall = {}

display(Markdown(f"""
### Latest run

`{run_dir}`

- success: `{overall.get('pc_success', 'n/a')}`
- episodes: `{overall.get('n_episodes', 'n/a')}`
- eval seconds: `{overall.get('eval_s', 'n/a')}`
"""))

videos = sorted((run_dir / "videos").glob("**/*.mp4"))
print("rollout videos:", len(videos))
if videos:
    display(Video(str(videos[0]), embed=True, width=720))

if CAPTURE_ACTIVATIONS:
    analysis_dir = run_dir / "analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)
    if GENERATE_DIAGNOSTIC_VIDEO:
        analysis_cmd = [
            str(PYTHON), "scripts/make_pi05_analysis_video.py",
            "--run", str(run_dir),
            "--task-id", str(ANALYSIS_TASK_ID),
            "--preview-frame", "30",
        ]
        subprocess.run(analysis_cmd, cwd=LOCAL_REPO, check=True)
        previews = sorted(analysis_dir.glob("*_frame0030.png"))
        analysis_videos = sorted(analysis_dir.glob("*.mp4"))
        if previews:
            display(Markdown("### Four-panel diagnostic preview"))
            display(Image(filename=str(previews[-1]), width=1000))
        if analysis_videos:
            display(Video(str(analysis_videos[-1]), embed=True, width=900))

    report_cmd = [
        str(PYTHON), "scripts/make_pi05_colab_report.py",
        "--run", str(run_dir),
        "--task-id", str(ANALYSIS_TASK_ID),
        "--episode", "0",
        "--n-action-steps", "10",
        "--max-rows", str(REPORT_MAX_ROWS),
    ]
    subprocess.run(report_cmd, cwd=LOCAL_REPO, check=True)
    report_dir = analysis_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_colab_report"
    manifest_path = report_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_manifest.json"
    manifest = json.loads(manifest_path.read_text())

    display(Markdown("""
### Granular Investigation Report

The interactive report lets you switch between activation family, metric, layer, and an optional action overlay without rendering every layer image inline. The chunk matrix gives one row per policy call: simulator third-person frame, the two camera tensors fed to Pi0.5, labeled 10-step action heatmap, action summary, signed expert-layer delta, signed expert denoise delta, and the strongest action/layer links for that chunk. The action-to-layer correlation image shows which action features co-vary with expert-layer activation deltas across chunks.
"""))
    interactive_path = manifest.get("interactive_html")
    if interactive_path and Path(interactive_path).exists():
        display(HTML(Path(interactive_path).read_text()))
    else:
        for key, width in [
            ("chunk_matrix", 1600),
            ("family_heatmaps", 1100),
            ("expert_layers_grid", 1100),
        ]:
            path = manifest.get(key)
            if path and Path(path).exists():
                display(Markdown(f"#### {key.replace('_', ' ').title()}"))
                display(Image(filename=str(path), width=width))

    layer_graphs = [Path(p) for p in manifest.get("expert_layer_graphs", [])]
    if DISPLAY_INDIVIDUAL_LAYER_GRAPHS and layer_graphs:
        display(Markdown("#### Individual Expert Layer Graphs"))
        for path in layer_graphs[:max(0, int(LAYER_GRAPH_LIMIT))]:
            if path.exists():
                display(Image(filename=str(path), width=900))

    rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
else:
    display(Markdown("Activation report skipped because `CAPTURE_ACTIVATIONS=False`."))


# Transcoder latent summary.
transcoder_events = run_dir / "transcoder_capture" / "events.jsonl"
if RUN_MODE in {"probe", "replace"}:
    if not transcoder_events.exists():
        display(Markdown("### Transcoder Latents\n\nNo transcoder latent file was found for this run."))
    else:
        import collections
        import numpy as np
        import matplotlib.pyplot as plt

        events = []
        for line in transcoder_events.read_text().splitlines():
            if line.strip():
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        latent_events = [event for event in events if event.get("type") == "transcoder_latent"]
        display(Markdown(f"### Transcoder Latents\n\nCaptured `{len(latent_events)}` transcoder calls in `{transcoder_events}`."))

        if latent_events:
            by_layer = collections.defaultdict(list)
            for event in latent_events:
                by_layer[int(event["layer"])].append(event)
            layers = sorted(by_layer)
            chunks = sorted({int(event["chunk"]) for event in latent_events})
            chunk_to_col = {chunk: i for i, chunk in enumerate(chunks)}
            heat = np.full((len(layers), len(chunks)), np.nan, dtype=np.float32)
            l0_heat = np.full_like(heat, np.nan)
            for r, layer in enumerate(layers):
                grouped = collections.defaultdict(list)
                for event in by_layer[layer]:
                    grouped[int(event["chunk"])].append(event)
                for chunk, group in grouped.items():
                    c = chunk_to_col[chunk]
                    heat[r, c] = float(np.mean([item["l1_mean"] for item in group]))
                    l0_heat[r, c] = float(np.mean([item["l0_mean"] for item in group]))

            fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=140)
            im0 = axes[0].imshow(heat, aspect="auto", interpolation="nearest")
            axes[0].set_title("Mean latent L1 by layer/chunk")
            axes[0].set_xlabel("Policy chunk")
            axes[0].set_ylabel("Action-expert layer")
            axes[0].set_yticks(range(len(layers)))
            axes[0].set_yticklabels(layers)
            fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

            im1 = axes[1].imshow(l0_heat, aspect="auto", interpolation="nearest")
            axes[1].set_title("Mean active-feature count by layer/chunk")
            axes[1].set_xlabel("Policy chunk")
            axes[1].set_ylabel("Action-expert layer")
            axes[1].set_yticks(range(len(layers)))
            axes[1].set_yticklabels(layers)
            fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
            fig.tight_layout()
            latent_plot = run_dir / "transcoder_capture" / "latent_layer_chunk_summary.png"
            fig.savefig(latent_plot, bbox_inches="tight")
            plt.close(fig)
            display(Image(filename=str(latent_plot), width=1200))

            top_events = sorted(latent_events, key=lambda event: event.get("max_value", 0.0), reverse=True)[:10]
            rows = ["| rank | chunk | layer | timestep | max value | top index |", "|---:|---:|---:|---|---:|---|"]
            for rank, event in enumerate(top_events, start=1):
                top = event.get("top", [])
                top_index = top[0]["index"] if top else []
                timestep = event.get("timestep", [])
                rows.append(
                    f"| {rank} | {event.get('chunk')} | {event.get('layer')} | `{timestep}` | "
                    f"{event.get('max_value', 0.0):.4f} | `{top_index}` |"
                )
            display(Markdown("\n".join(rows)))
else:
    display(Markdown("### Transcoder Latents\n\nSkipped because `RUN_MODE='original'`."))


In [ ]:
import json, collections
import numpy as np
import matplotlib.pyplot as plt

events_path = LATEST_RUN / "transcoder_capture" / "events.jsonl"
layer = 7

by_t_chunk = collections.defaultdict(list)
for line in events_path.read_text().splitlines():
    ev = json.loads(line)
    if ev.get("type") != "transcoder_latent" or int(ev["layer"]) != layer:
        continue
    t = round(float((ev.get("timestep") or [0])[0]), 1)
    by_t_chunk[(t, int(ev["chunk"]))].append(ev["l1_mean"])

ts = sorted({k[0] for k in by_t_chunk})
chunks = sorted({k[1] for k in by_t_chunk})
heat = np.array([[np.mean(by_t_chunk[(t, c)]) for c in chunks] for t in ts])

plt.imshow(heat, aspect="auto")
plt.yticks(range(len(ts)), ts)
plt.xticks(range(len(chunks)), chunks)
plt.ylabel("t")
plt.xlabel("chunk")
plt.title(f"TC_{layer} L1  (t × chunk)")
plt.colorbar()
plt.show()

In [ ]:
import json
from pathlib import Path

events_path = LATEST_RUN / "transcoder_capture" / "events.jsonl"
layer = 7
t_target = 0.1
t_tol = 0.05

rows = []
for line in events_path.read_text().splitlines():
    ev = json.loads(line)
    if ev.get("type") != "transcoder_latent":
        continue
    if int(ev["layer"]) != layer:
        continue
    ts = ev.get("timestep") or []
    t = float(ts[0]) if ts else None
    if t is None or abs(t - t_target) > t_tol:
        continue
    rows.append(ev)

print(f"TC_{layer} @ t≈{t_target}: {len(rows)} records")
for ev in rows[:5]:
    top = ev.get("top", [])[:5]
    print(
        f"chunk={ev['chunk']} t={ev['timestep']} "
        f"L0={ev['l0_mean']:.1f} L1={ev['l1_mean']:.3f} "
        f"top={[(x['index'], x['value']) for x in top]}"
    )

## Transcoder Equivalence Report

After both sanity branches have produced runs, this section builds the cross-run comparison report. It also runs a paired same-observation action-equivalence probe to estimate how much action error the transcoder replacement adds before simulator feedback diverges.


In [ ]:
# @title Run Transcoder Equivalence Metrics And Sanity Comparison

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown

comparison_base_dirs = [
    LOCAL_OUTPUT_ROOT.parent,
    DRIVE_OUTPUTS / "eval/pi05_libero",
]
comparison_out = LOCAL_REPO / "outputs/eval/pi05_libero/sanity-comparison" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
comparison_out.mkdir(parents=True, exist_ok=True)


def stream_command(label: str, cmd: list[str], log_path: Path, *, allow_failure: bool = False):
    print(f"\n== {label} ==", flush=True)
    print("$", " ".join(map(str, cmd)), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("wb") as log_handle:
        process = subprocess.Popen(
            [str(item) for item in cmd],
            cwd=LOCAL_REPO,
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=0,
        )
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            log_handle.write(chunk)
            log_handle.flush()
            buffer = getattr(sys.stdout, "buffer", None)
            if buffer is not None:
                buffer.write(chunk)
                buffer.flush()
            else:
                sys.stdout.write(chunk.decode("utf-8", errors="replace"))
                sys.stdout.flush()
        returncode = process.wait()
    if returncode != 0 and not allow_failure:
        lines = log_path.read_text(errors="replace").splitlines()
        print(f"\n{label} failed with exit code {returncode}", flush=True)
        print(f"--- last {min(80, len(lines))} lines from {log_path} ---", flush=True)
        for line in lines[-80:]:
            print(line[-500:], flush=True)
        raise subprocess.CalledProcessError(returncode, cmd)
    return returncode

comparison_cmd = [
    str(PYTHON), "scripts/compare_pi05_transcoder_sanity_runs.py",
    "--output-dir", str(comparison_out),
    "--n-action-steps", "10",
]
for base_dir in comparison_base_dirs:
    comparison_cmd.extend(["--base-dir", str(base_dir)])
if SANITY_COMPARE_MAKE_VIDEO:
    comparison_cmd.append("--make-video")

comparison_return = stream_command(
    "Compare original/probe vs replace sanity runs",
    comparison_cmd,
    comparison_out / "sanity_comparison.log",
    allow_failure=True,
)
if comparison_return != 0:
    display(Markdown(
        "### Cross-run comparison not ready\n\n"
        "The comparison script could not find both `original-probe-sanity` and `replace-sanity` runs yet. "
        "Run both sanity notebooks first, then rerun this cell."
    ))
else:
    report_html = comparison_out / "sanity_comparison_report.html"
    summary_json = comparison_out / "sanity_comparison_summary.json"
    if report_html.exists():
        display(Markdown("### Closed-Loop Sanity Comparison"))
        display(HTML(report_html.read_text()))
    if summary_json.exists():
        summary = json.loads(summary_json.read_text())
        print("Closed-loop verdict:", summary.get("verdict"), flush=True)

if not TRANSCODER_CHECKPOINT:
    raise RuntimeError("Resolve the transcoder checkpoint before running paired action equivalence.")

equiv_out = LOCAL_REPO / "outputs/eval/pi05_libero/action-equivalence" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
equiv_cmd = [
    str(PYTHON), "scripts/eval_pi05_transcoder_action_equivalence.py",
    "--policy-path", PROBE_POLICY_PATH,
    "--checkpoint", TRANSCODER_CHECKPOINT,
    "--output-dir", str(equiv_out),
    "--episodes", ACTION_EQUIV_EPISODES,
    "--batch-size", str(ACTION_EQUIV_BATCH_SIZE),
    "--num-workers", "0",
    "--num-inference-steps", str(ACTION_EQUIV_NUM_INFERENCE_STEPS),
    "--n-action-steps", "10",
    "--device", "cuda",
    "--policy-dtype", "bfloat16",
    "--control-batches", str(ACTION_EQUIV_CONTROL_BATCHES),
]
if str(ACTION_EQUIV_MAX_BATCHES).strip():
    equiv_cmd.extend(["--max-batches", str(ACTION_EQUIV_MAX_BATCHES).strip()])

stream_command("Run paired same-observation action equivalence", equiv_cmd, equiv_out / "action_equivalence.log")
equiv_summary_path = equiv_out / "action_equivalence_summary.json"
if equiv_summary_path.exists():
    equiv_summary = json.loads(equiv_summary_path.read_text())
    equiv_metrics = equiv_summary.get("metrics", {})
    control_metrics = equiv_summary.get("control_metrics") or {}
    display(Markdown("### Paired Same-Observation Action Equivalence"))

    def _stat(metrics, key, field="mean"):
        entry = metrics.get(key) or {}
        value = entry.get(field)
        return "n/a" if value is None else f"{value:.5g}"

    headline_rows = [
        ("paired observations", equiv_metrics.get("pairs", 0)),
        ("executed-window relative L2 (mean)", _stat(equiv_metrics, "executed_rel_l2")),
        ("executed-window relative L2 (p95)", _stat(equiv_metrics, "executed_rel_l2", "p95")),
        ("executed-window RMSE (mean)", _stat(equiv_metrics, "executed_rmse")),
        ("chunk cosine similarity (mean)", _stat(equiv_metrics, "cosine")),
        ("share within 5% relative L2", equiv_metrics.get("executed_rel_l2_le_0.05")),
        ("determinism floor, relative L2", _stat(control_metrics, "executed_rel_l2") if control_metrics else "control disabled"),
    ]
    display(HTML(
        "<table><tr><th>metric</th><th>value</th></tr>"
        + "".join(f"<tr><td>{name}</td><td>{value}</td></tr>" for name, value in headline_rows)
        + "</table>"
    ))
    print(
        "Read the relative-L2 numbers against the determinism floor: only the excess is "
        "error introduced by the transcoder substitution.",
        flush=True,
    )
    print(json.dumps(equiv_metrics, indent=2)[:5000], flush=True)

rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
print("Comparison report dir:", comparison_out)
print("Action equivalence dir:", equiv_out)
print("Synced reports under:", DRIVE_OUTPUTS / "eval/pi05_libero")


## Counterfactual Object Perturbation

Recolor one object at a frozen simulator state and measure which transcoder
features respond. Physics is never stepped between the two renders, so the
observations differ only in that object's pixels.

Run once with `COUNTERFACTUAL_TARGET` empty to list the scene's objects, then
set the target and rerun.


In [ ]:
# @title Run Counterfactual Object-Perturbation Probe

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image as IPyImage

if "stream_command" not in globals():
    def stream_command(label: str, cmd: list, log_path: Path, *, allow_failure: bool = False):
        print(f"\n== {label} ==", flush=True)
        print("$", " ".join(map(str, cmd)), flush=True)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("wb") as log_handle:
            process = subprocess.Popen(
                [str(item) for item in cmd], cwd=LOCAL_REPO, env=os.environ.copy(),
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
            )
            assert process.stdout is not None
            while True:
                chunk = process.stdout.read(4096)
                if not chunk:
                    break
                log_handle.write(chunk); log_handle.flush()
                buffer = getattr(sys.stdout, "buffer", None)
                if buffer is not None:
                    buffer.write(chunk); buffer.flush()
                else:
                    sys.stdout.write(chunk.decode("utf-8", errors="replace")); sys.stdout.flush()
            returncode = process.wait()
        if returncode != 0 and not allow_failure:
            raise subprocess.CalledProcessError(returncode, cmd)
        return returncode

cf_out = LOCAL_REPO / "outputs/probes/counterfactual" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
cf_out.mkdir(parents=True, exist_ok=True)

cf_cmd = [
    str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
    "--policy-path", PROBE_POLICY_PATH,
    "--output-dir", str(cf_out),
    "--suite", PROBE_SUITE,
    "--task-id", str(PROBE_TASK_ID),
    "--seed", str(PROBE_SEED),
    "--device", "cuda",
    "--policy-dtype", "bfloat16",
]
if ENABLE_LANGFUSE_TRACE:
    cf_cmd.append("--trace-langfuse")
    os.environ.setdefault(
        "PI05_LANGFUSE_SESSION_ID", f"counterfactual-{PROBE_SUITE}-{PROBE_TASK_ID}"
    )

target = COUNTERFACTUAL_TARGET.strip()
if not target:
    # Discovery pass: no policy forward passes, no checkpoint needed.
    cf_cmd.append("--list-objects")
else:
    if not TRANSCODER_CHECKPOINT:
        raise RuntimeError("Resolve the transcoder checkpoint before running the counterfactual probe.")
    cf_cmd += [
        "--checkpoint", str(TRANSCODER_CHECKPOINT),
        "--target", target,
        "--perturbation", COUNTERFACTUAL_PERTURBATION,
        "--color", COUNTERFACTUAL_COLOR,
        "--dose", COUNTERFACTUAL_DOSE,
        "--states", str(COUNTERFACTUAL_STATES),
        "--state-stride", str(COUNTERFACTUAL_STATE_STRIDE),
    ]
    if COUNTERFACTUAL_PLACEBO_TARGET.strip():
        cf_cmd += ["--placebo-target", COUNTERFACTUAL_PLACEBO_TARGET.strip()]

stream_command("Counterfactual object-perturbation probe", cf_cmd, cf_out / "counterfactual.log")

if not target:
    display(Markdown(
        "### Scene objects listed\n\n"
        "Copy a body name into `COUNTERFACTUAL_TARGET` in the Controls cell, then rerun this cell. "
        "Set `COUNTERFACTUAL_PLACEBO_TARGET` to a different object to get the specificity control."
    ))
else:
    summary_path = cf_out / "counterfactual_summary.json"
    if summary_path.exists():
        cf_summary = json.loads(summary_path.read_text())
        rows = cf_summary.get("measurements", [])
        display(Markdown("### Counterfactual Activation Response"))
        display(HTML(
            "<table><tr><th>state</th><th>kind</th><th>target</th><th>dose</th>"
            "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
            + "".join(
                "<tr><td>{state_index}</td><td>{kind}</td><td>{target}</td><td>{dose}</td>"
                "<td>{px:.5f}</td><td>{lat}</td><td>{act:.5g}</td></tr>".format(
                    state_index=r["state_index"], kind=r["kind"], target=r["target"], dose=r["dose"],
                    px=r["pixel"]["changed_pixel_fraction"],
                    lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                         else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                    act=r["action_relative_l2"],
                )
                for r in rows
            )
            + "</table>"
        ))
        print("Read every row against the 'null' rows: that is the nondeterminism floor.", flush=True)

    for png in sorted((cf_out / "images").glob("*.png"))[:12]:
        display(Markdown(f"**{png.name}**"))
        display(IPyImage(filename=str(png)))

shapes_path = cf_out / "observation_shapes.json"
if shapes_path.exists():
    shapes = json.loads(shapes_path.read_text())
    display(Markdown("### Observation Shapes Sent To The Policy"))
    display(HTML(
        "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
        + "".join(
            f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
            for k, v in sorted(shapes.get("tensor_shapes", {}).items())
        )
        + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
          f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
    ))
    print("task:", repr(shapes.get("task")), flush=True)

if "rsync_tree" in globals() and "DRIVE_OUTPUTS" in globals():
    rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
else:
    print("Drive not mounted; artifacts stay local at", cf_out, flush=True)
print("Counterfactual probe dir:", cf_out)


## Experimental: One-Chunk Plaintext Prompt Probe

This cell is for mechanistic interpretability, not benchmark scoring.

It resets one LIBERO task environment, captures the two camera views and robot state, replaces the task language with `PROBE_LANGUAGE`, and asks Pi0.5 for one 50-action chunk. It saves the input images and action chunk so you can compare plaintext prompts against internal activations and action outputs.

Caveat: Pi0.5-LIBERO was fine-tuned on LIBERO-style task prompts. Low-level prompts like `move +x` or `close gripper` may be out of distribution. Use this to inspect behavior, not to claim official benchmark success.

In [ ]:
# @title Run One-Chunk Plaintext Prompt Probe

import os
import subprocess
import sys
from pathlib import Path

probe_out = LOCAL_REPO / "outputs/probes" / PROBE_SUITE / f"task_{PROBE_TASK_ID}"
probe_out.mkdir(parents=True, exist_ok=True)
probe_log = probe_out / "probe.log"

probe_code = r"""
import json
import os
import shutil
import site
import traceback
from pathlib import Path

import cv2
import numpy as np
import torch
from huggingface_hub import snapshot_download

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot.scripts.lerobot_eval import preprocess_observation

policy_path = os.environ.get("PROBE_POLICY_PATH", "lerobot/pi05_libero_finetuned")
suite = os.environ["PROBE_SUITE"]
task_id = int(os.environ["PROBE_TASK_ID"])
prompt = os.environ["PROBE_LANGUAGE"]
out_dir = Path(os.environ["PROBE_OUT"])
out_dir.mkdir(parents=True, exist_ok=True)


def find_libero_root() -> Path:
    roots = [Path(path) for path in site.getsitepackages()]
    user_site = site.getusersitepackages()
    if user_site:
        roots.append(Path(user_site))
    for root in roots:
        candidate = root / "libero" / "libero"
        if candidate.exists():
            return candidate
    raise RuntimeError("Could not find installed LIBERO package path")


def ensure_libero_assets() -> None:
    libero_root = find_libero_root()
    assets_dir = libero_root / "assets"
    required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
    if required.exists():
        print("libero_assets OK", required, flush=True)
        return
    offline = os.environ.get("HF_HUB_OFFLINE") == "1"
    cache_dir = os.environ.get("HF_HUB_CACHE") or str(Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser() / "hub")
    print("LIBERO assets missing; installing lerobot/libero-assets before prompt probe", flush=True)
    snapshot = Path(snapshot_download(
        repo_id="lerobot/libero-assets",
        repo_type="dataset",
        cache_dir=cache_dir,
        local_files_only=offline,
        token=os.environ.get("HF_TOKEN") or None,
    ))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snapshot.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("libero_assets OK", required, flush=True)


def write_image(path: Path, image) -> None:
    arr = image
    if hasattr(arr, "detach"):
        arr = arr.detach().cpu().numpy()
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[0] in (1, 3):
        arr = np.moveaxis(arr, 0, -1)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    if arr.max() <= 1.5:
        arr = np.clip(arr, 0, 1) * 255
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    cv2.imwrite(str(path), cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))


env = None
try:
    print(f"Prompt probe policy_path={policy_path}", flush=True)
    print(f"Prompt probe suite={suite} task_id={task_id}", flush=True)
    print(f"Prompt: {prompt}", flush=True)
    ensure_libero_assets()

    policy_cfg = PreTrainedConfig.from_pretrained(
        policy_path,
        cache_dir=os.environ["HF_HUB_CACHE"],
        local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
    )
    # Some Pi0.5 configs point pretrained_path back to the base repo.
    # Force the probe to use the same fine-tuned checkpoint path as lerobot-eval.
    policy_cfg.pretrained_path = Path(policy_path)
    policy_cfg.device = "cuda"
    policy_cfg.dtype = "bfloat16"
    policy_cfg.compile_model = False
    policy_cfg.gradient_checkpointing = False
    policy_cfg.n_action_steps = 10

    env_cfg = LiberoEnv(task=suite, task_ids=[task_id])
    envs = make_env(env_cfg, n_envs=1, use_async_envs=False)
    env = envs[suite][task_id]

    policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg, rename_map={})
    policy.eval()

    preprocessor, postprocessor = make_pre_post_processors(
        policy_cfg=policy_cfg,
        pretrained_path=policy_path,
        preprocessor_overrides={
            "device_processor": {"device": str(policy.config.device)},
            "rename_observations_processor": {"rename_map": {}},
        },
    )
    env_preprocessor, _env_postprocessor = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)

    seed = int(os.environ.get("PROBE_SEED", "1000"))
    obs, info = env.reset(seed=[seed])
    raw_render = env.envs[0].render() if hasattr(env, "envs") else env.call("render")[0]
    write_image(out_dir / "render.png", raw_render)

    obs = preprocess_observation(obs)
    obs["task"] = [prompt]
    for key, value in obs.items():
        if key.startswith("observation.images."):
            write_image(out_dir / f"{key.replace('.', '_')}.png", value[0])

    batch = env_preprocessor(obs)
    batch = preprocessor(batch)
    with torch.inference_mode():
        actions = policy.predict_action_chunk(batch).detach().cpu().float()[0]

    payload = {
        "suite": suite,
        "task_id": task_id,
        "seed": seed,
        "prompt": prompt,
        "action_shape": list(actions.shape),
        "first_10_actions": actions[:10].tolist(),
        "mean_abs_per_dim": actions.abs().mean(dim=0).tolist(),
    }
    (out_dir / "action_chunk.json").write_text(json.dumps(payload, indent=2))
    print(json.dumps(payload, indent=2)[:4000], flush=True)
    print("Prompt probe artifacts saved to", out_dir, flush=True)
except Exception:
    traceback.print_exc()
    raise
finally:
    if env is not None:
        try:
            env.close()
        except Exception as exc:
            print(f"env.close failed: {type(exc).__name__}: {exc}", flush=True)
"""

env = os.environ.copy()
env.update({
    "PROBE_POLICY_PATH": globals().get("PROBE_POLICY_PATH", "lerobot/pi05_libero_finetuned"),
    "PROBE_SUITE": PROBE_SUITE,
    "PROBE_TASK_ID": str(PROBE_TASK_ID),
    "PROBE_LANGUAGE": PROBE_LANGUAGE,
    "PROBE_SEED": str(PROBE_SEED),
    "PROBE_OUT": str(probe_out),
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
})

def run_probe_with_log():
    cmd = [str(PYTHON), "-u", "-c", probe_code]
    print("Running prompt probe:", " ".join(cmd[:3]), "<probe_code>", flush=True)
    print("Probe log:", probe_log, flush=True)
    with probe_log.open("wb") as log_handle:
        process = subprocess.Popen(
            cmd,
            cwd=LOCAL_REPO,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=0,
        )
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            log_handle.write(chunk)
            log_handle.flush()
            buffer = getattr(sys.stdout, "buffer", None)
            if buffer is not None:
                buffer.write(chunk)
                buffer.flush()
            else:
                sys.stdout.write(chunk.decode("utf-8", errors="replace"))
                sys.stdout.flush()
        returncode = process.wait()
    if returncode != 0:
        print(f"\nPrompt probe failed with exit code {returncode}", flush=True)
        if probe_log.exists():
            lines = probe_log.read_text(errors="replace").splitlines()
            print(f"--- last {min(80, len(lines))} probe log lines ---", flush=True)
            for line in lines[-80:]:
                print(line[-500:], flush=True)
        raise subprocess.CalledProcessError(returncode, cmd)

run_probe_with_log()

outputs_dir = LOCAL_REPO / "outputs"
if "rsync_tree" in globals():
    rsync_tree(outputs_dir, DRIVE_OUTPUTS, reset=False)
else:
    subprocess.run(["rsync", "-a", "--human-readable", "--info=progress2", "--stats", f"{outputs_dir}/", f"{DRIVE_OUTPUTS}/"], check=True)
print("Probe saved to:", probe_out)
print("Probe log:", probe_log)
print("Probe synced to:", DRIVE_OUTPUTS)


## Aggregate Transcoder Feature Flow Probe

This runs the transcoder feature-discovery pass across LIBERO dataset frames, then renders a layer-by-layer flow report. The output folder is `outputs/features/pi05_libero/transcoder-probe` and is synced back to Drive.


In [ ]:
# @title Run Aggregate Transcoder Feature Flow Probe

import os
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image

if not TRANSCODER_CHECKPOINT:
    raise RuntimeError("Resolve the transcoder checkpoint before running the aggregate transcoder probe.")

feature_dir = LOCAL_REPO / "outputs/features/pi05_libero" / TRANSCODER_PROBE_NAME
if RESET_TRANSCODER_PROBE_OUTPUT and feature_dir.exists():
    shutil.rmtree(feature_dir)
feature_dir.mkdir(parents=True, exist_ok=True)

probe_env = os.environ.copy()
probe_env.update({
    "PYTHON": str(PYTHON),
    "PYTHONPATH": f"{LOCAL_REPO / 'src'}:{LOCAL_REPO / 'scripts'}:" + probe_env.get("PYTHONPATH", ""),
    "PI05_LANGFUSE_TRACE": "1" if ENABLE_LANGFUSE_TRACE else "0",
    "PI05_LANGFUSE_MEDIA": "1" if LANGFUSE_ATTACH_IMAGES else "0",
    "PI05_LANGFUSE_MAX_IMAGES": str(LANGFUSE_MAX_IMAGES),
    "PI05_LANGFUSE_MAX_MEDIA_BYTES": str(LANGFUSE_MAX_MEDIA_BYTES),
    "PI05_LANGFUSE_CALLER": LANGFUSE_CALLER.strip() or "colab",
    "PI05_LANGFUSE_TENANT_ID": LANGFUSE_TENANT_ID.strip() or "libero-local",
    "PI05_LANGFUSE_SESSION_ID": f"{TRANSCODER_PROBE_NAME}-{SUITE}-{PROBE_TASK_ID}",
    "PI05_LANGFUSE_ENVIRONMENT": LANGFUSE_ENVIRONMENT.strip() or "development",
    "PI05_LANGFUSE_TAGS": LANGFUSE_TAGS.strip(),
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
})


def run_with_log(label: str, cmd: list[str], log_path: Path):
    print(f"\n== {label} ==", flush=True)
    print("$", " ".join(map(str, cmd)), flush=True)
    with log_path.open("wb") as log_handle:
        process = subprocess.Popen(
            [str(item) for item in cmd],
            cwd=LOCAL_REPO,
            env=probe_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=0,
        )
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            log_handle.write(chunk)
            log_handle.flush()
            buffer = getattr(sys.stdout, "buffer", None)
            if buffer is not None:
                buffer.write(chunk)
                buffer.flush()
            else:
                sys.stdout.write(chunk.decode("utf-8", errors="replace"))
                sys.stdout.flush()
        returncode = process.wait()
    if returncode != 0:
        lines = log_path.read_text(errors="replace").splitlines() if log_path.exists() else []
        print(f"\n{label} failed with exit code {returncode}", flush=True)
        print(f"--- last {min(80, len(lines))} lines from {log_path} ---", flush=True)
        for line in lines[-80:]:
            print(line[-500:], flush=True)
        raise subprocess.CalledProcessError(returncode, cmd)


collect_cmd = [
    str(PYTHON), "scripts/collect_pi05_transcoder_features.py",
    "--policy-path", PROBE_POLICY_PATH,
    "--checkpoint", TRANSCODER_CHECKPOINT,
    "--output-dir", str(feature_dir),
    "--episodes", FEATURE_PROBE_EPISODES,
    "--batch-size", str(FEATURE_PROBE_BATCH_SIZE),
    "--num-workers", str(FEATURE_PROBE_NUM_WORKERS),
    "--collection-mode", FEATURE_PROBE_COLLECTION_MODE,
    "--num-inference-steps", str(FEATURE_PROBE_NUM_INFERENCE_STEPS),
    "--noise-samples", str(FEATURE_PROBE_NOISE_SAMPLES),
    "--top-k", str(FEATURE_PROBE_TOP_K),
    "--top-m-active", str(FEATURE_PROBE_TOP_M_ACTIVE),
    "--device", "cuda",
    "--policy-dtype", "bfloat16",
]
if str(FEATURE_PROBE_MAX_BATCHES).strip():
    collect_cmd.extend(["--max-batches", str(FEATURE_PROBE_MAX_BATCHES).strip()])
if ENABLE_LANGFUSE_TRACE:
    collect_cmd.append("--langfuse-trace")
run_with_log("Collect transcoder features", collect_cmd, feature_dir / "collect.log")

feature_report_cmd = [
    str(PYTHON), "scripts/make_pi05_feature_report.py",
    "--feature-dir", str(feature_dir),
    "--max-features", str(FEATURE_REPORT_MAX_FEATURES),
    "--top-examples", str(FEATURE_REPORT_TOP_EXAMPLES),
]
if FEATURE_REPORT_SAVE_THUMBNAILS:
    feature_report_cmd.extend(["--save-thumbnails", "--device", "cuda", "--policy-dtype", "bfloat16"])
run_with_log("Render feature browser", feature_report_cmd, feature_dir / "feature_report.log")

flow_report_cmd = [
    str(PYTHON), "scripts/make_pi05_transcoder_flow_report.py",
    "--feature-dir", str(feature_dir),
    "--checkpoint", TRANSCODER_CHECKPOINT,
    "--top-features-per-layer", str(FEATURE_FLOW_TOP_FEATURES_PER_LAYER),
]
if ENABLE_LANGFUSE_TRACE:
    flow_report_cmd.append("--langfuse-trace")
run_with_log("Render layer flow report", flow_report_cmd, feature_dir / "flow_report.log")

flow_html = feature_dir / "transcoder_flow_report.html"
flow_svg = feature_dir / "transcoder_flow_chart.svg"
feature_html = feature_dir / "feature_report.html"

if flow_html.exists():
    display(Markdown("### Transcoder Layer Flow"))
    display(HTML(flow_html.read_text()))
elif flow_svg.exists():
    display(Image(filename=str(flow_svg), width=1400))

if feature_html.exists():
    display(Markdown("### Feature Browser"))
    display(HTML(feature_html.read_text()))

rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
probe_archive = DRIVE_ARCHIVES / f"{TRANSCODER_PROBE_NAME}.tar"
timed("write transcoder probe archive to Drive", lambda: create_tar(feature_dir, probe_archive))
print("Transcoder probe saved to:", feature_dir)
print("Transcoder probe synced to:", DRIVE_OUTPUTS / "features/pi05_libero" / TRANSCODER_PROBE_NAME)
print("Transcoder probe archive:", probe_archive)


## Run All Checklist

Before sharing:

1. Save a copy of this notebook in the restricted Drive folder.
2. Share the notebook only with `programmer908@gmail.com` and other approved accounts.
3. Share `DRIVE_ROOT` only with the same approved accounts.
4. For first cache fill, keep `CACHE_TRANSFER_MODE="archive"`, set `ALLOW_AUTH_REFRESH=True`, and store `HF_TOKEN` only in private Colab Secrets.
5. After `archives/hf_home.tar` exists, leave `ALLOW_AUTH_REFRESH=True` or set it to `False`; the archive path is used first unless `FORCE_AUTH_REFRESH=True`.
6. Set Runtime -> Change runtime type -> GPU -> L4 or A100, and use a high-RAM runtime. T4 is intentionally rejected because Pi0.5 usually exits 137 while loading the policy.
7. Runtime -> Run all.

Outputs are persisted back to `DRIVE_ROOT/outputs` after every run, including rollout videos, diagnostic videos, chunk matrices, activation heatmaps, and individual layer graphs. Model/assets caches are persisted as tar archives under `DRIVE_ROOT/archives` so future sessions start faster.